# COMP64702 RAG Coursework
## Notebook 2: RAG Pipeline

Implements the full RAG pipeline:
- **Chunking** — sentence-boundary splitting with overlap; short/stub entries filtered
- **Embedding** — `all-MiniLM-L6-v2` dense embeddings with title prefix
- **Retrieval** — Hybrid BM25 + dense with Reciprocal Rank Fusion (RRF)
- **Generation** — Qwen2.5-0.5B-Instruct with adaptive prompting
- **Output** — saves `test_outputs.json` and `test_outputs_baseline.json`

**Files:**
- Input corpus: `Background_Corpus_Final_Combined.json`
- Input queries: `benchmark_input_only.json`
- Output: `test_outputs.json`

In [ ]:
# Install dependencies (run once)
!pip install sentence-transformers transformers rank-bm25 accelerate -q

In [ ]:
# Imports
import json
import re
import numpy as np
from pathlib import Path
from dataclasses import dataclass
from typing import List, Dict

from sentence_transformers import SentenceTransformer
from transformers import AutoTokenizer, AutoModelForCausalLM
from rank_bm25 import BM25Okapi
import torch

print('All imports OK')


In [ ]:
# File paths
DATA_DIR = Path('.')

corpus_path = DATA_DIR / 'background_corpus.json'
query_path  = DATA_DIR / 'benchmark_input_only.json'
output_path = DATA_DIR / 'test_outputs.json'

with open(corpus_path, 'r', encoding='utf-8') as f:
    corpus = json.load(f)

with open(query_path, 'r', encoding='utf-8') as f:
    query_payload = json.load(f)

queries = query_payload['queries']
print(f'Corpus: {len(corpus)} documents')
print(f'Queries: {len(queries)}')


In [ ]:
# Data structures
@dataclass
class Chunk:
    doc_id:  str
    title:   str
    section: str
    text:    str

@dataclass
class RetrievalResult:
    chunk: Chunk
    score: float


In [ ]:
# ── Chunking ──────────────────────────────────────────────────────────────
# Strategy:
#   1. Documents with sections → one chunk per section
#   2. Documents with only full_text → sentence-boundary chunks (~400 words)
#      with 50-word overlap to preserve cross-boundary context
#   3. Skip scaffold/meta entries and short stubs (<500 chars) that add noise

CHUNK_WORDS     = 400
OVERLAP_WORDS   = 50
MIN_ENTRY_CHARS = 500   # filter out very short/stub entries

def split_into_chunks(text: str, max_words=CHUNK_WORDS, overlap=OVERLAP_WORDS) -> List[str]:
    """Split text into overlapping sentence-boundary chunks."""
    sentences = re.split(r'(?<=[.!?])\s+', text.strip())
    chunks, current, current_len = [], [], 0
    for sent in sentences:
        words = sent.split()
        if current_len + len(words) > max_words and current:
            chunks.append(' '.join(current))
            overlap_sents, overlap_len = [], 0
            for s in reversed(current):
                wlen = len(s.split())
                if overlap_len + wlen > overlap:
                    break
                overlap_sents.insert(0, s)
                overlap_len += wlen
            current, current_len = overlap_sents, overlap_len
        current.append(sent)
        current_len += len(words)
    if current:
        chunks.append(' '.join(current))
    return chunks


def chunk_documents(corpus_docs: list) -> List[Chunk]:
    chunks = []
    for doc in corpus_docs:
        doc_id    = doc.get('doc_id', '')
        title     = doc.get('title', '')
        sections  = doc.get('sections', [])
        full_text = doc.get('full_text', '').strip()

        # Skip scaffold / meta planning entries
        if 'scaffold' in doc_id or 'scope' in doc_id:
            continue

        # Skip very short entries — too little content to be useful
        if not sections and len(full_text) < MIN_ENTRY_CHARS:
            continue

        if sections:
            for sec in sections:
                heading = sec.get('heading', 'Section')
                text = ' '.join(sec.get('content', [])).strip()
                if len(text.split()) >= 20:
                    chunks.append(Chunk(doc_id=doc_id, title=title,
                                        section=heading, text=text))
        else:
            word_count = len(full_text.split())
            if word_count <= CHUNK_WORDS:
                chunks.append(Chunk(doc_id=doc_id, title=title,
                                    section='Full text', text=full_text))
            else:
                for i, part in enumerate(split_into_chunks(full_text)):
                    chunks.append(Chunk(doc_id=doc_id, title=title,
                                        section=f'Part {i+1}', text=part))
    return chunks


chunks = chunk_documents(corpus)
print(f'Total chunks: {len(chunks)}')
print(f'Avg chunk length (words): {sum(len(c.text.split()) for c in chunks) // len(chunks)}')

from collections import Counter
wiki_chunks = [c for c in chunks if c.doc_id.startswith('wiki')]
print(f'Wiki chunks in index: {len(wiki_chunks)} (only long wiki entries kept)')
print(f'Chunks under 30 words: {sum(1 for c in chunks if len(c.text.split()) < 30)}')


In [ ]:
# ── Embedding ─────────────────────────────────────────────────────────────
embedding_model = SentenceTransformer('all-MiniLM-L6-v2')
print('Model loaded:', embedding_model.get_sentence_embedding_dimension(), 'dims')

chunk_texts = [f"{c.title}: [{c.section}] {c.text}" for c in chunks]

chunk_embeddings = embedding_model.encode(
    chunk_texts,
    convert_to_numpy=True,
    batch_size=64,
    show_progress_bar=True
)

norms = np.linalg.norm(chunk_embeddings, axis=1, keepdims=True) + 1e-12
chunk_embeddings_norm = chunk_embeddings / norms
print(f'Embeddings shape: {chunk_embeddings_norm.shape}')


In [ ]:
# ── BM25 ──────────────────────────────────────────────────────────────────
def tokenize(text: str) -> List[str]:
    return re.findall(r'\b[a-z0-9]+\b', text.lower())

tokenized_chunks = [tokenize(t) for t in chunk_texts]
bm25 = BM25Okapi(tokenized_chunks)
print('BM25 index built over', len(tokenized_chunks), 'chunks')


In [ ]:
# ── Dense retrieval ───────────────────────────────────────────────────────
def dense_retrieve(query: str, top_k: int = 10) -> List[RetrievalResult]:
    q_emb = embedding_model.encode([query], convert_to_numpy=True)[0]
    q_emb = q_emb / (np.linalg.norm(q_emb) + 1e-12)
    sims = chunk_embeddings_norm @ q_emb
    top_idx = np.argsort(sims)[::-1][:top_k]
    return [RetrievalResult(chunk=chunks[i], score=float(sims[i])) for i in top_idx]


In [ ]:
# ── BM25 retrieval ────────────────────────────────────────────────────────
def bm25_retrieve(query: str, top_k: int = 10) -> List[RetrievalResult]:
    scores = bm25.get_scores(tokenize(query))
    top_idx = np.argsort(scores)[::-1][:top_k]
    return [RetrievalResult(chunk=chunks[i], score=float(scores[i])) for i in top_idx]


In [ ]:
# ── Hybrid retrieval (RRF) ────────────────────────────────────────────────
# Reciprocal Rank Fusion (Cormack et al., 2009): RRF_K=60 is standard.
# Diversification: max 2 chunks per doc to avoid one document dominating.

RRF_K       = 60
TOP_K       = 5
MAX_PER_DOC = 2

def hybrid_retrieve(query: str, top_k: int = TOP_K) -> List[RetrievalResult]:
    dense_res = dense_retrieve(query, top_k=10)
    bm25_res  = bm25_retrieve(query,  top_k=10)

    score_map: Dict[str, dict] = {}
    for rank, res in enumerate(dense_res, 1):
        key = f"{res.chunk.doc_id}||{res.chunk.section}||{res.chunk.text[:80]}"
        score_map.setdefault(key, {'chunk': res.chunk, 'score': 0.0})
        score_map[key]['score'] += 1.0 / (RRF_K + rank)

    for rank, res in enumerate(bm25_res, 1):
        key = f"{res.chunk.doc_id}||{res.chunk.section}||{res.chunk.text[:80]}"
        score_map.setdefault(key, {'chunk': res.chunk, 'score': 0.0})
        score_map[key]['score'] += 1.0 / (RRF_K + rank)

    merged = sorted(score_map.values(), key=lambda x: x['score'], reverse=True)

    selected, doc_counts = [], {}
    for item in merged:
        did = item['chunk'].doc_id
        if doc_counts.get(did, 0) >= MAX_PER_DOC:
            continue
        selected.append(RetrievalResult(chunk=item['chunk'], score=item['score']))
        doc_counts[did] = doc_counts.get(did, 0) + 1
        if len(selected) == top_k:
            break

    return selected


def baseline_retrieve(query: str, top_k: int = TOP_K) -> List[RetrievalResult]:
    """Dense-only retrieval — used as baseline in evaluation."""
    return dense_retrieve(query, top_k=top_k)


In [ ]:
# ── Smoke test retrieval ──────────────────────────────────────────────────
test_query = 'What is biryani and which region is it originally associated with?'
test_results = hybrid_retrieve(test_query)

print(f'Top {len(test_results)} chunks retrieved:\n')
for i, r in enumerate(test_results, 1):
    print(f'{i}. [{r.chunk.doc_id} | {r.chunk.section}]  score={r.score:.4f}')
    print(f'   {r.chunk.text[:120]}...')
    print()


In [ ]:
# ── Load Qwen2.5-0.5B-Instruct ───────────────────────────────────────────
MODEL_ID = 'Qwen/Qwen2.5-0.5B-Instruct'

tokenizer = AutoTokenizer.from_pretrained(MODEL_ID)
model = AutoModelForCausalLM.from_pretrained(
    MODEL_ID,
    torch_dtype='auto',
    device_map='auto'
)
model.eval()
print(f'Model loaded on: {next(model.parameters()).device}')


In [ ]:
# ── Prompt builder ────────────────────────────────────────────────────────
MAX_CONTEXT_CHARS = 1800

def build_prompt(query: str, retrieved: List[RetrievalResult]) -> List[dict]:
    context_parts = []
    for r in retrieved:
        text = r.chunk.text[:MAX_CONTEXT_CHARS]
        context_parts.append(
            f"[Source: {r.chunk.title} | {r.chunk.section}]\n{text}"
        )
    context = '\n\n---\n\n'.join(context_parts)

    q_lower = query.lower()
    if any(w in q_lower for w in ['differ', 'difference', 'distinguish', 'compare', 'vs', 'versus']):
        task_hint = 'This is a comparison question. Identify the key differences supported by the sources.'
    elif any(w in q_lower for w in ['how is', 'how are', 'how do', 'prepared', 'made', 'cook']):
        task_hint = 'This is a procedural question. Describe the main steps or method supported by the sources.'
    elif any(w in q_lower for w in ['what is', 'what are', 'define', 'describe']):
        task_hint = 'This is a factual question. Provide a clear, direct answer supported by the sources.'
    else:
        task_hint = 'Answer the question using only information from the sources below.'

    return [
        {
            'role': 'system',
            'content': (
                'You are a knowledgeable South Asian cuisine assistant. '
                'Answer questions using ONLY the information in the retrieved sources. '
                'Do NOT add facts not present in the sources. '
                'Be concise and accurate. '
                'Do not mention "retrieved sources" or "context" in your answer.'
            )
        },
        {
            'role': 'user',
            'content': (
                f'Retrieved sources:\n{context}\n\n'
                f'Question: {query}\n\n'
                f'{task_hint} '
                f'Write your answer in 2 to 3 complete sentences.'
            )
        }
    ]


In [ ]:
# ── Generation ────────────────────────────────────────────────────────────
def postprocess(text: str) -> str:
    text = text.strip()
    for prefix in ['Answer:', 'Response:', 'A:']:
        if text.startswith(prefix):
            text = text[len(prefix):].strip()
    # Drop any incomplete final sentence (no terminal punctuation)
    sentences = re.split(r'(?<=[.!?])\s+', text)
    if len(sentences) > 1 and not text.rstrip().endswith(('.', '!', '?')):
        text = ' '.join(sentences[:-1])
    return re.sub(r'\s+', ' ', text).strip()


def generate(messages: list) -> str:
    prompt_text = tokenizer.apply_chat_template(
        messages, tokenize=False, add_generation_prompt=True
    )
    inputs = tokenizer([prompt_text], return_tensors='pt').to(model.device)

    with torch.no_grad():
        output_ids = model.generate(
            **inputs,
            max_new_tokens=200,        # high enough to avoid mid-sentence cutoffs
            do_sample=False,
            temperature=None,
            top_p=None,
            repetition_penalty=1.1,
            pad_token_id=tokenizer.eos_token_id
        )

    new_ids = output_ids[0][inputs['input_ids'].shape[-1]:]
    return postprocess(tokenizer.decode(new_ids, skip_special_tokens=True))


In [ ]:
# ── Single query test ─────────────────────────────────────────────────────
sample_query    = queries[0]['query']
sample_results  = hybrid_retrieve(sample_query)
sample_messages = build_prompt(sample_query, sample_results)
sample_answer   = generate(sample_messages)

print('QUERY:  ', sample_query)
print('ANSWER: ', sample_answer)
print()
print('Retrieved docs:')
for r in sample_results:
    print(f'  - {r.chunk.doc_id} | {r.chunk.section}')


In [ ]:
# ── Full inference loop ───────────────────────────────────────────────────
results = []

for i, item in enumerate(queries):
    query_id = item['query_id']
    query    = item['query']

    retrieved = hybrid_retrieve(query, top_k=TOP_K)
    messages  = build_prompt(query, retrieved)
    response  = generate(messages)

    results.append({
        'query_id': query_id,
        'query':    query,
        'response': response,
        'retrieved_context': [
            {
                'doc_id': r.chunk.doc_id,
                # Store FULL chunk text — faithfulness metric reads this field.
                # Truncating causes artificially low faithfulness scores.
                'text': r.chunk.text
            }
            for r in retrieved
        ]
    })

    if (i + 1) % 10 == 0:
        print(f'  {i+1}/{len(queries)} done')

print(f'\nGenerated {len(results)} answers')


In [ ]:
# ── Save test_outputs.json ────────────────────────────────────────────────
with open(output_path, 'w', encoding='utf-8') as f:
    json.dump({'results': results}, f, indent=2, ensure_ascii=False)

print(f'Saved {len(results)} results to {output_path}')


In [ ]:
# ── Baseline (dense-only) for evaluation comparison ──────────────────────
baseline_results = []

for i, item in enumerate(queries):
    query_id = item['query_id']
    query    = item['query']

    retrieved = baseline_retrieve(query, top_k=TOP_K)
    messages  = build_prompt(query, retrieved)
    response  = generate(messages)

    baseline_results.append({
        'query_id': query_id,
        'query':    query,
        'response': response,
        'retrieved_context': [
            {'doc_id': r.chunk.doc_id, 'text': r.chunk.text}
            for r in retrieved
        ]
    })

    if (i + 1) % 10 == 0:
        print(f'  {i+1}/{len(queries)} done')

baseline_path = DATA_DIR / 'test_outputs_baseline.json'
with open(baseline_path, 'w', encoding='utf-8') as f:
    json.dump({'results': baseline_results}, f, indent=2, ensure_ascii=False)

print(f'Baseline saved to {baseline_path}')


In [ ]:
# ── Interactive demo ──────────────────────────────────────────────────────
# Use this cell during the live demonstration.

demo_query     = input('Enter your question: ')
demo_retrieved = hybrid_retrieve(demo_query)
demo_messages  = build_prompt(demo_query, demo_retrieved)
demo_answer    = generate(demo_messages)

print('\n── ANSWER ─────────────────────────────────────────────────')
print(demo_answer)

print('\n── RETRIEVED CONTEXT ──────────────────────────────────────')
for i, r in enumerate(demo_retrieved, 1):
    print(f'\n{i}. [{r.chunk.doc_id} | {r.chunk.section}]  (score: {r.score:.5f})')
    print(r.chunk.text[:300])
